# PD scorecard and final conclusions

This notebook converts the fitted `P(good)` logistic model into an illustrative 300–850 scorecard, then scores the held-out accounts. PD remains `1 - P(good)` throughout.

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

processed = Path('../data/processed')
with open(processed / 'pd_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open(processed / 'model_feature_names.pkl', 'rb') as f:
    model_feature_names = pickle.load(f)
with open(processed / 'reference_categories.pkl', 'rb') as f:
    reference_categories = pickle.load(f)
with open(processed / 'validation_predictions.pkl', 'rb') as f:
    validation_predictions = pickle.load(f)
with open(processed / 'model_inputs_test.pkl', 'rb') as f:
    inputs_test_all = pickle.load(f)

## Reconstruct the coefficient table

Reference categories have zero coefficients because they were omitted during estimation. Adding them back makes every mutually exclusive category available for score calculation.

In [ ]:
estimated = pd.DataFrame({'feature': model_feature_names, 'coefficient': model.coef_.ravel()})
intercept = pd.DataFrame({'feature': ['Intercept'], 'coefficient': [model.intercept_[0]]})
references = pd.DataFrame({'feature': reference_categories, 'coefficient': 0.0})
scorecard = pd.concat([intercept, estimated, references], ignore_index=True)
if scorecard['feature'].duplicated().any():
    raise ValueError('Scorecard feature names must be unique')
scorecard['original_feature'] = scorecard['feature'].str.split(':', n=1).str[0]
scorecard

## Score scale and coefficient extremes

The original scorecard maps the lowest and highest feasible coefficient sums to 300 and 850. The intercept is part of each account score.

In [ ]:
min_score, max_score = 300, 850
family_extremes = scorecard.groupby('original_feature')['coefficient'].agg(['min', 'max'])
min_sum_coef = family_extremes['min'].sum()
max_sum_coef = family_extremes['max'].sum()
scale = (max_score - min_score) / (max_sum_coef - min_sum_coef)
pd.DataFrame({'minimum_coefficient_sum': [min_sum_coef], 'maximum_coefficient_sum': [max_sum_coef], 'scale': [scale]})

## Preliminary category scores

The intercept receives the base-score transformation; category coefficients are linearly rescaled. This preserves the arithmetic of the original scorecard while avoiding chained assignment.

In [ ]:
scorecard['score_calculation'] = scorecard['coefficient'] * scale
intercept_mask = scorecard['feature'].eq('Intercept')
scorecard.loc[intercept_mask, 'score_calculation'] = ((scorecard.loc[intercept_mask, 'coefficient'] - min_sum_coef) * scale + min_score)
scorecard['score_preliminary'] = scorecard['score_calculation'].round().astype(int)
scorecard['rounding_difference'] = scorecard['score_preliminary'] - scorecard['score_calculation']
scorecard[['feature', 'coefficient', 'score_calculation', 'score_preliminary', 'rounding_difference']]

## Rounding reconciliation

The source adjusted hard-coded row numbers. Here the same reconciliation is derived from the selected minimum and maximum categories, so it remains correct if row order changes.

In [ ]:
scorecard['score_final'] = scorecard['score_preliminary']
min_rows = scorecard.groupby('original_feature')['score_final'].idxmin()
max_rows = scorecard.groupby('original_feature')['score_final'].idxmax()
preliminary_min = scorecard.loc[min_rows, 'score_final'].sum()
preliminary_max = scorecard.loc[max_rows, 'score_final'].sum()
scorecard.loc[min_rows.iloc[0], 'score_final'] += min_score - preliminary_min
scorecard.loc[max_rows.iloc[0], 'score_final'] += max_score - preliminary_max
final_min = scorecard.loc[scorecard.groupby('original_feature')['score_final'].idxmin(), 'score_final'].sum()
final_max = scorecard.loc[scorecard.groupby('original_feature')['score_final'].idxmax(), 'score_final'].sum()
pd.DataFrame({'preliminary_minimum': [preliminary_min], 'preliminary_maximum': [preliminary_max], 'final_minimum': [final_min], 'final_maximum': [final_max]})

## Score held-out accounts

Each account activates one category per feature family. Column alignment is explicit before the matrix multiplication.

In [ ]:
score_features = [f for f in scorecard['feature'] if f != 'Intercept']
missing = sorted(set(score_features) - set(inputs_test_all.columns))
if missing:
    raise KeyError(f'Scorecard columns missing from held-out matrix: {missing}')
account_matrix = inputs_test_all[score_features].copy()
account_matrix.insert(0, 'Intercept', 1)
account_matrix = account_matrix[scorecard['feature']]
account_scores = account_matrix.dot(scorecard.set_index('feature')['score_final'])
account_scores.name = 'score'
account_scores.head()

## Relate the score to P(good) and PD

The score is a linear translation of the fitted log-odds, so higher scores should correspond to higher `P(good)` and lower PD.

In [ ]:
X_test = inputs_test_all[model_feature_names]
p_good = model.predict_proba(X_test)[:, 1]
scored_accounts = pd.DataFrame({'score': account_scores.to_numpy(), 'p_good': p_good, 'pd': 1 - p_good, 'actual_good': validation_predictions['actual_good'].to_numpy()})
scored_accounts.sort_values('score').head()

In [ ]:
plt.scatter(scored_accounts['score'], scored_accounts['pd'], alpha=0.15)
plt.xlabel('Score')
plt.ylabel('PD = 1 - P(good)')
plt.title('Held-out score and PD')
plt.show()
scored_accounts[['score', 'p_good', 'pd']].corr()

## Example account interpretation

The median-score held-out account offers a concrete interpretation without selecting an unusually favorable or unfavorable case.

In [ ]:
example = scored_accounts.iloc[(scored_accounts['score'] - scored_accounts['score'].median()).abs().argsort().iloc[0]]
print(f"Example score: {example['score']:.0f}")
print(f"P(good): {example['p_good']:.3f}")
print(f"PD: {example['pd']:.3f}")
print(f"Observed held-out outcome (good=1): {example['actual_good']:.0f}")

## Final conclusions

The workflow preserves the original detailed category inventory, uses reference levels to estimate an interpretable logistic specification, and validates ranking on held-out accounts through ROC/AUC, Gini, and KS. Threshold accuracy is reported with its class-imbalance limitation. The 300–850 scorecard is an educational translation of the fitted historical model: it ranks this held-out historical sample but is not a lending decision system.

### Reproducibility note

Run notebooks 00 through 05 in order. This notebook only consumes the saved modeling contract and does not repeat upstream preparation or feature engineering.